<a href="https://colab.research.google.com/github/uyx7uy/ds2002-fa26/blob/main/2026-10-16%20%E2%80%94%20Walmart%20Warm-Up%20SKU%20Mess%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
pop_tarts = cat[cat["product_name"].str.lower().str.contains("pop", na=False)]

print("Number of Pop-Tart rows:", len(pop_tarts))
print(pop_tarts)

Number of Pop-Tart rows: 3
             sku          product_name         category
0          PT-12  pop-tarts strawberry  Breakfast Foods
1     SKU#459812   Strawberry Pop-Tart        Breakfast
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
strawberry = pop_tarts[
    pop_tarts["product_name"].str.lower().str.contains("strawberry", na=False)
]

other_pop_tarts = pop_tarts[
    ~pop_tarts["product_name"].str.lower().str.contains("strawberry", na=False)
]

print("Strawberry Pop-Tart SKUs and names:")
print(strawberry[["sku", "product_name"]])

print("\nPop-Tart rows that are not strawberry:")
print(other_pop_tarts[["sku", "product_name"]])

Strawberry Pop-Tart SKUs and names:
             sku          product_name
0          PT-12  pop-tarts strawberry
1     SKU#459812   Strawberry Pop-Tart
2  POPTART-STRAW  Pop-Tarts Strawberry

Pop-Tart rows that are not strawberry:
Empty DataFrame
Columns: [sku, product_name]
Index: []


Pop-Tart rows that are a different product: None were found in the loaded sample because all three rows are strawberry Pop-Tarts. Any other flavors in the full catalog should stay separate from the strawberry mapping.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
rows_before = len(cat)

canonical_sku = strawberry["sku"].iloc[0]
sku_mapping = {}

for sku in strawberry["sku"]:
    sku_mapping[sku] = canonical_sku

cat["canonical_sku"] = cat["sku"].replace(sku_mapping)

print(sku_mapping)
print(cat[["sku", "product_name", "canonical_sku"]])

{'PT-12': 'PT-12', 'SKU#459812': 'PT-12', 'POPTART-STRAW': 'PT-12'}
             sku          product_name canonical_sku
0          PT-12  pop-tarts strawberry         PT-12
1     SKU#459812   Strawberry Pop-Tart         PT-12
2  POPTART-STRAW  Pop-Tarts Strawberry         PT-12


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
print("Category counts before:")
print(cat["category"].value_counts(dropna=False))

cat["category"] = cat["category"].str.strip().str.title()

print("\nCategory counts after:")
print(cat["category"].value_counts(dropna=False))

Category counts before:
category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64

Category counts after:
category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64


Stripping spaces and using title case fixes formatting differences, but “Breakfast” and “Breakfast Foods” still remain separate labels even though they describe the same category.

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
cat["normalized_name"] = (
    cat["product_name"]
    .str.lower()
    .str.replace(r"[^a-z0-9\s]", "", regex=True)
    .str.split()
    .apply(sorted)
    .str.join(" ")
)

sku_counts = cat.groupby("normalized_name")["sku"].nunique()
duplicate_names = sku_counts[sku_counts > 1].index

candidates = cat[cat["normalized_name"].isin(duplicate_names)]

print(
    candidates[["normalized_name", "sku", "product_name"]]
    .sort_values("normalized_name")
)

       normalized_name            sku          product_name
0  poptarts strawberry          PT-12  pop-tarts strawberry
2  poptarts strawberry  POPTART-STRAW  Pop-Tarts Strawberry


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
assert cat["canonical_sku"].isna().sum() == 0
assert len(cat) == rows_before

print("The canonical SKU column has no missing values.")
print("The number of rows stayed the same.")

The canonical SKU column has no missing values.
The number of rows stayed the same.


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a) : Q5 found only strawberry Pop-Tart variants because the notebook loaded the three-row sample. It contains no other product families, so we still need the full product_catalog.csv to name the two families this question requires.

b) : Consolidation becomes a business decision when similar product names might hide differences that matter for sales or inventory. For example, I am not sure whether “Strawberry Pop-Tart” and “Pop-Tarts Strawberry” have the same package size and quantity because those details are missing. I would ask the category manager to confirm before treating them as the same product.